# 02 · Polymarket — full World Cup data run

**Purpose.** Build the canonical PM dataset family for every WC-related
market we hold data on: metadata, outcomes/tokens, prices, order books,
trades, volume/liquidity — as tidy tables at event / market / outcome /
snapshot / book-level / trade grain.

**Sources** (real, no fabrication):
* `data/pm_orderflow.db` — production capture: ~2.09M trades + ~1.4k
  markets (slugs, questions, outcomes, token ids, volume, liquidity).
* Live Gamma / CLOB / Data-API — attempted per cell; each failure is
  recorded with its reason (the PM route needs the VPN on this MacBook).

**Contents**: [markets](#markets) · [classification](#classes) ·
[match events](#events) · [trades](#trades) · [live layers](#live) ·
[coverage](#coverage) · [findings](#findings)

In [1]:
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
JB = next(q for q in [_here, *_here.parents] if (q / "lib" / "bootstrap.py").exists())
sys.path.insert(0, str(JB))

import json
import pandas as pd
import polars as pl
import lib.bootstrap as bt
import lib.config as cfg
import lib.storage as st
import lib.pmdata as pm
import lib.matching as mt
import lib.ids as ids

manifest = bt.run_manifest("02_polymarket_full_run")
p = cfg.load_params()
N_LIVE_BOOKS = 12      # order books to snapshot live (open markets only)
N_HISTORY_TOKENS = 4   # tokens to pull dense CLOB price history for
print(f"offline={p.offline}")

offline=False


/Users/andrewdoherty/Desktop/Coding/World Cup Alpha/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


<a id="markets"></a>
## 1 · Market universe (bronze)
Straight from the production capture DB — read-only.

In [2]:
mk = pm.orderflow_markets()
st.save_dataset(mk, "bronze", "pm_markets",
                inputs=["repo:data/pm_orderflow.db#pm_markets"], notebook="02")
prof = st.profile_frame(mk, "pm_markets", unique_keys=["condition_id"])
print(f"{prof.attrs['rows']} markets, duplicate condition_ids: {prof.attrs['duplicate_keys']}")
prof

1447 markets, duplicate condition_ids: 0


,column,dtype,null_rate
0,condition_id,String,0.0000
1,event_slug,String,0.0000
2,market_slug,String,0.0000
3,question,String,0.0000
4,event_title,String,0.0000
5,category,String,0.0000
6,team,String,0.0504
7,outcomes,String,0.0000
8,token_ids,String,0.0000
9,closed,Int64,0.0000


<a id="classes"></a>
## 2 · Classification
`category` is PRODUCTION's own classification (carried in the DB);
`lib.matching.classify_pm_market` maps it to our canonical vocabulary
with text fallbacks for live-Gamma rows that lack it.

In [3]:
classes = pl.DataFrame({
    "class": [mt.classify_pm_market(r) for r in mk.to_dicts()],
    "condition_id": mk["condition_id"], "volume": mk["volume"],
    "closed": mk["closed"]})
(classes.group_by("class")
 .agg(pl.len().alias("n_markets"),
      pl.col("volume").sum().round(0).alias("total_usd_volume"),
      (1 - pl.col("closed").mean()).round(3).alias("open_frac"))
 .sort("total_usd_volume", descending=True)
 .to_pandas())

,class,n_markets,total_usd_volume,open_frac
0,outright,60,3.796644e+09,0.483
1,match_1x2,282,2.049319e+09,0.096
2,advance,240,3.760070e+07,0.262
3,other_future,805,2.166623e+07,0.216
4,group_winner,60,1.708724e+07,0.017


<a id="events"></a>
## 3 · Canonical match events + outcome (token) table — silver
PM match markets come as one Yes/No market per outcome
(`fifwc-<a>-<b>-<date>-{a,b,draw}`); `pm_match_events` reassembles the
1X2 with slug order = home first. Settlement: **90 minutes** (the PM
match markets settle on regulation, matching sportsbook 1X2 — advancement
markets, ET+pens, are a different table and can never be confused).

In [4]:
ev = mt.pm_match_events(mk)
st.save_dataset(ev, "silver", "pm_match_events",
                inputs=["bronze/pm_markets"], notebook="02")
n_groups = (classes.filter(pl.col("class") == "match_1x2").height) // 3
print(f"assembled {ev.height} matches (from ~{n_groups} slug groups; "
      f"unassembled groups lack a home/away leg and are counted, not hidden)")
ev.tail(8).to_pandas()

assembled 91 matches (from ~94 slug groups; unassembled groups lack a home/away leg and are counted, not hidden)


,event_slug,home,away,kickoff_utc,cid_home,cid_away,cid_draw,n_outcome_markets
0,fifwc-tur-usa-2026-06-25,Türkiye,United States,2026-06-26 02:00:00+00:00,0x81cef42df3d25aecf0bf7e96c344202e86f447f2f3ba...,0x32552863fe8014da2613037970661f0d49f415c86a88...,0xdc34b1a1507fb269c41fb4750880a7774d396f00b76e...,3
1,fifwc-ury-cvi-2026-06-21,Uruguay,Cabo Verde,2026-06-21 22:00:00+00:00,0x08d56195e88cedb87eeff396bd35e8b67d031e586169...,0x21454470816337a6dadbea2140070a57ec40146b8e65...,0x2d8a8c670430061537dd6cb60c0fee0d0e0a4a3430fd...,3
2,fifwc-ury-esp-2026-06-26,Uruguay,Spain,2026-06-27 00:00:00+00:00,0x61b80a4a7a325eba229ccfa35aaef9cf5879a4ad38be...,0xe322faca2a534900680db54e3a4349a61427d347b6f9...,0xd0fa27da024261ba5edb5e9eda33651a224fb5a2137d...,3
3,fifwc-usa-aus-2026-06-19,United States,Australia,2026-06-19 19:00:00+00:00,0x7ebb07134794576c154f06fbaa1cc61a5292d58e280c...,0xb1fca83dd9b3046c8db3ae0de2c8d56ee5d17d3aad54...,0xffa05800960fb8b1532c0f3f99f51b4996c9b447b0b1...,3
4,fifwc-usa-bel-2026-07-06,United States,Belgium,2026-07-07 00:00:00+00:00,0xf5a0c2d5b6bf0a7b9899aaf4c6237b08e1dbf8d17430...,0x02103f049a5fe127c4ea03c9531772e82f3b9f5e9f5f...,0x982941d6030e30ed0f56043622c52d71db557f6ef277...,3
5,fifwc-usa-bih-2026-07-01,United States,Bosnia and Herzegovina,2026-07-02 00:00:00+00:00,0xe9d96f957f3f5e4ffa0e920087edf967c4cc353e9d1a...,0xd8aff2490e3a808f2cbef7970698f72762f8090b7856...,0xe2671b2075027e789dd1066e9976fd4515b4db921f84...,3
6,fifwc-usa-par-2026-06-12,United States,Paraguay,2026-06-13 01:00:00+00:00,0x26c10c27cde58e2d6214d65ab562d8142d34723b3b6e...,0x86f102a31cf4b12dbcf31ae8c62c526c0b538d5b7eb1...,0x3774cdf17e49143a4f6e10812df8b06113731493842f...,3
7,fifwc-uzb-col-2026-06-17,Uzbekistan,Colombia,2026-06-18 02:00:00+00:00,0x0ef9d17aff09c873dc47a93cfb13c42ab6274b4493e0...,0x2f00e48e914ba6606867a6347e90c48cc1fd4da66cc1...,0x3e0e84f19abf090dae4490caf1d7f74d5cbaa5d49fa0...,3


In [5]:
# outcome/token grain: one row per (market, outcome), preserving source ids
out_rows = []
for r in mk.to_dicts():
    outcomes = json.loads(r["outcomes"]) if r["outcomes"] else []
    tokens = json.loads(r["token_ids"]) if r["token_ids"] else []
    for i, o in enumerate(outcomes):
        out_rows.append({
            "condition_id": r["condition_id"], "outcome_index": i,
            "outcome": o, "token_id": tokens[i] if i < len(tokens) else None,
            "market_class": mt.classify_pm_market(r),
            "question": r["question"], "closed": bool(r["closed"]),
            "resolved_outcome_index": r["resolved_outcome_index"],
            "volume_usd": r["volume"], "liquidity_usd": r["liquidity"]})
outcomes_df = pl.DataFrame(out_rows)
st.save_dataset(outcomes_df, "silver", "pm_outcomes",
                inputs=["bronze/pm_markets"], notebook="02")
st.profile_frame(outcomes_df, "pm_outcomes",
                 unique_keys=["condition_id", "outcome_index"])

,column,dtype,null_rate
0,condition_id,String,0.0000
1,outcome_index,Int64,0.0000
2,outcome,String,0.0000
3,token_id,String,0.0000
4,market_class,String,0.0000
5,question,String,0.0000
6,closed,Boolean,0.0000
7,resolved_outcome_index,Int64,0.2032
8,volume_usd,Float64,0.0000
9,liquidity_usd,Float64,0.0000


<a id="trades"></a>
## 4 · Trades (bronze, 2M+ rows — Polars lazy)
The full capture, tidied to trade grain with UTC timestamps. This is REAL
executed flow, so prices/volumes here are executable history, not quotes.
Beware the production caveat: the data-api caps history at offset 3000,
so this DB (incrementally captured since May) is the ONLY full history.

In [6]:
lf = pm.orderflow_trades()          # LazyFrame — filter before collecting
trades_summary = (lf
    .group_by("condition_id")
    .agg(pl.len().alias("n_trades"), pl.col("usd").sum().alias("usd_vol"),
         pl.col("ts").min().alias("first_ts"), pl.col("ts").max().alias("last_ts"))
    .collect())
print(f"{int(trades_summary['n_trades'].sum()):,} trades across "
      f"{trades_summary.height} markets")
top = (trades_summary.join(mk.select("condition_id", "question"), on="condition_id")
       .sort("usd_vol", descending=True).head(10))
top.select("question", "n_trades", "usd_vol").to_pandas()

2,086,434 trades across 1300 markets


,question,n_trades,usd_vol
0,Will Canada win on 2026-06-28?,4951,1.764450e+07
1,Will Belgium win on 2026-06-15?,6395,1.725545e+07
2,Will Congo DR win the 2026 FIFA World Cup?,6590,1.703112e+07
3,Will United States win on 2026-06-25?,7580,1.540141e+07
4,Will USA win the 2026 FIFA World Cup?,9622,1.462798e+07
5,Will Belgium win on 2026-07-01?,6034,1.437293e+07
6,Will Germany win on 2026-06-20?,5947,1.409749e+07
7,Will United States win on 2026-06-12?,5644,1.381268e+07
8,Will France win on 2026-06-30?,5252,1.283125e+07
9,Will IR Iran win on 2026-06-15?,6455,1.242202e+07


In [7]:
# Persist the tidy trade table once (bronze). ~2M rows → zstd parquet.
trades_path = st.dataset_path("bronze", "pm_trades")
if not trades_path.exists():
    tidy = lf.collect()
    st.save_dataset(tidy, "bronze", "pm_trades",
                    inputs=["repo:data/pm_orderflow.db#pm_trades"], notebook="02",
                    note="full production orderflow capture, trade grain")
    print(f"bronze pm_trades written: {tidy.height:,} rows, "
          f"{trades_path.stat().st_size/1e6:.0f} MB")
    del tidy
else:
    print(f"bronze pm_trades already built "
          f"({trades_path.stat().st_size/1e6:.0f} MB) — idempotent skip")
st.save_dataset(trades_summary, "silver", "pm_trade_summary",
                inputs=["bronze/pm_trades"], notebook="02")

bronze pm_trades already built (118 MB) — idempotent skip


PosixPath('/Users/andrewdoherty/Desktop/Coding/World Cup Alpha/jupyter bet/data/silver/pm_trade_summary.parquet')

<a id="live"></a>
## 5 · Live layers — Gamma metadata, CLOB books, price history, holders
Each attempted independently; failures recorded (VPN route required).
Books give bid/ask/mid/microprice/spread + $ depth (full level data kept
in raw); price history is the dense CLOB source (production-verified
back to May 30).

In [8]:
gamma_status = "skipped (offline)"
if not p.offline:
    try:
        gamma_events, snaps = pm.gamma_wc_events()
        gamma_status = f"OK — {len(gamma_events)} WC events → {snaps[-1]}"
        gm = pl.DataFrame([{k: (json.dumps(v) if isinstance(v, (dict, list)) else v)
                            for k, v in e.items()} for e in gamma_events],
                          infer_schema_length=None)
        st.save_dataset(gm, "bronze", "pm_gamma_events", notebook="02",
                        inputs=list(snaps))
    except pm.PMUnavailable as e:
        gamma_status = f"UNREACHABLE: {e}"
    except Exception as e:  # noqa: BLE001 — record, don't crash the run
        gamma_status = f"ERROR: {type(e).__name__}: {e}"
print("gamma:", gamma_status)

gamma: OK — 2 WC events → polymarket/gamma_events_wc/20260704T132729Z__ae3c5c0f3f


In [9]:
# Live order books for the most active OPEN markets
quotes_rows, book_status = [], "skipped (offline)"
if not p.offline:
    open_active = (trades_summary
        .join(mk.filter(pl.col("closed") == 0).select("condition_id", "question"),
              on="condition_id")
        .sort("last_ts", descending=True).head(N_LIVE_BOOKS))
    ok = fail = 0
    for r in open_active.to_dicts():
        toks = (outcomes_df.filter(pl.col("condition_id") == r["condition_id"])
                .to_dicts())
        for t in toks:
            if not t["token_id"]:
                continue
            try:
                book = pm.clob_book(t["token_id"])
            except pm.PMUnavailable:
                book = None
            if not book:
                fail += 1
                continue
            m = pm.book_metrics(book)
            quotes_rows.append({
                "condition_id": r["condition_id"], "question": r["question"],
                "outcome": t["outcome"], "token_id": t["token_id"],
                "snapshot_utc": bt.utcnow_iso(), **m})
            ok += 1
    book_status = f"{ok} books captured, {fail} failed/empty"
print("clob books:", book_status)
quotes = pl.DataFrame(quotes_rows) if quotes_rows else pl.DataFrame()
if quotes.height:
    st.save_dataset(quotes, "silver", "pm_quotes_live", notebook="02",
                    note="live top-of-book snapshot at run time")
    display(quotes.head(8).to_pandas())

clob books: 12 books captured, 12 failed/empty


,condition_id,question,outcome,token_id,snapshot_utc,best_bid,best_ask,mid,microprice,spread,bid_sz_top,ask_sz_top,depth_bid_5c_usd,depth_ask_5c_usd
0,0xe99cc59f32b10d23acf196d1a0e8264ea30fca198428...,Will Colombia win the 2026 FIFA World Cup?,Yes,9880339017552145671265367828047492063793459623...,2026-07-04T13:27:35Z,0.030,0.031,0.0305,0.030953,0.001,128346.74,6354.35,4.392006e+04,5.424540e+04
1,0xe99cc59f32b10d23acf196d1a0e8264ea30fca198428...,Will Colombia win the 2026 FIFA World Cup?,No,6682696535116667515588751516730608630741233222...,2026-07-04T13:27:36Z,0.969,0.970,0.9695,0.969047,0.001,6354.35,128346.74,1.339620e+06,7.177051e+06
2,0xf9dfc7eae628396f2f00cd1c53329b021a0f13615c69...,Will USA be eliminated in the Quarterfinals of...,Yes,6214036442962346112081305080905148514204724298...,2026-07-04T13:27:37Z,0.331,0.337,0.3340,0.335591,0.006,220.00,67.53,1.745163e+03,5.965431e+02
3,0xf9dfc7eae628396f2f00cd1c53329b021a0f13615c69...,Will USA be eliminated in the Quarterfinals of...,No,1687858767182509328675280335029386476208443236...,2026-07-04T13:27:38Z,0.663,0.669,0.6660,0.664409,0.006,67.53,220.00,1.096047e+03,4.105037e+03
4,0x3a26ca6425e2d98f14935670bc22cdb0744defc6f6d8...,Will Switzerland win the 2026 FIFA World Cup?,Yes,6213191364851514826646381669430603139453965659...,2026-07-04T13:27:38Z,0.008,0.009,0.0085,0.008956,0.001,5439521.36,250181.06,7.974734e+04,4.435854e+04
5,0x3a26ca6425e2d98f14935670bc22cdb0744defc6f6d8...,Will Switzerland win the 2026 FIFA World Cup?,No,4531527275011679183650401366602958351753290831...,2026-07-04T13:27:39Z,0.991,0.992,0.9915,0.991044,0.001,250181.06,5439521.36,2.743991e+06,1.733569e+07
6,0x30d55d8124ee1e12dabe89201badc45669b81dff69e4...,Will Brazil win the 2026 FIFA World Cup?,Yes,2757653331728340157775899938464276040592173849...,2026-07-04T13:27:40Z,0.064,0.065,0.0645,0.064097,0.001,5340.03,49884.56,1.997154e+05,3.784563e+05
7,0x30d55d8124ee1e12dabe89201badc45669b81dff69e4...,Will Brazil win the 2026 FIFA World Cup?,No,5298671877490835733041265348647134744981889350...,2026-07-04T13:27:41Z,0.935,0.936,0.9355,0.935903,0.001,49884.56,5340.03,3.830753e+06,4.501274e+06


In [10]:
# Dense CLOB price history for a few active tokens (trajectory source)
hist_status = "skipped (offline)"
hist_frames = []
if not p.offline and quotes.height:
    for t in quotes.head(N_HISTORY_TOKENS).to_dicts():
        try:
            h = pm.clob_price_history(t["token_id"])
            if h.height:
                hist_frames.append(h.with_columns(
                    pl.lit(t["question"]).alias("question"),
                    pl.lit(t["outcome"]).alias("outcome")))
        except pm.PMUnavailable as e:
            hist_status = f"UNREACHABLE: {e}"
            break
    else:
        hist_status = f"OK — {len(hist_frames)} token histories"
if hist_frames:
    hist = pl.concat(hist_frames)
    st.save_dataset(hist, "bronze", "pm_price_history", notebook="02")
print("clob price history:", hist_status)

clob price history: OK — 0 token histories


In [11]:
# Holders / open interest (data-api; live only, sample)
holders_status = "skipped (offline)"
if not p.offline and quotes.height:
    cid = quotes["condition_id"][0]
    try:
        hl = pm.data_api_holders(cid)
        holders_status = f"OK — {len(hl)} holder rows for {cid[:10]}…"
    except pm.PMUnavailable as e:
        holders_status = f"UNREACHABLE: {e}"
print("holders:", holders_status)

holders: OK — 2 holder rows for 0xe99cc59f…


### Chart — real price + volume trajectory (from captured trades)

In [12]:
import matplotlib.pyplot as plt
import lib.plotting as plot
import lib.convergence as cv

sample = top.head(1).to_dicts()[0]
tr = pm.orderflow_trades(condition_ids=[sample["condition_id"]]).collect()
flow = cv.trade_flow_metrics(tr, bucket="1h")
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(11, 6))
for outcome, grp in tr.group_by("outcome"):
    g = grp.sort("ts_utc")
    ax1.plot(g["ts_utc"], g["price"], lw=0.8, label=str(outcome[0]) if isinstance(outcome, tuple) else str(outcome))
ax1.set_ylabel("trade price")
ax1.set_title(f"{sample['question']} — {int(sample['n_trades']):,} real trades")
ax1.legend(loc="best", fontsize=8)
ax2.bar(flow["ts_utc"].to_list(), flow["usd_vol"].to_list(), width=0.04)
ax2.set_ylabel("$ volume / h")
plot.ts_axis(ax2)
plot.save_fig(fig, "02_sample_market_trajectory",
              note=f"source: pm_orderflow.db · n={tr.height} trades · built {manifest['run_utc']}")
plt.show()

/var/folders/04/sqz7rdl96l5csx4yrrfnjxgm0000gn/T/ipykernel_48654/3331363624.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<a id="coverage"></a>
## 6 · Coverage — observed vs unavailable (honesty table)

In [13]:
coverage = pd.DataFrame([
    {"data": "markets metadata", "status": f"observed — {mk.height} markets (capture DB)"},
    {"data": "outcomes/tokens", "status": f"observed — {outcomes_df.height} rows"},
    {"data": "trades (full history)", "status": f"observed — {int(trades_summary['n_trades'].sum()):,} trades"},
    {"data": "live gamma metadata", "status": gamma_status},
    {"data": "live order books (depth/spread)", "status": book_status},
    {"data": "dense price history (CLOB)", "status": hist_status},
    {"data": "holders / OI", "status": holders_status},
    {"data": "HISTORICAL order books", "status": "unavailable — books were never captured historically; "
     "historical spread/depth cannot be reconstructed and are not"},
])
plot.save_table(coverage, "02_pm_coverage")
coverage

,data,status
0,markets metadata,observed — 1447 markets (capture DB)
1,outcomes/tokens,observed — 2894 rows
2,trades (full history),"observed — 2,086,434 trades"
3,live gamma metadata,OK — 2 WC events → polymarket/gamma_events_wc/...
4,live order books (depth/spread),"12 books captured, 12 failed/empty"
5,dense price history (CLOB),OK — 0 token histories
6,holders / OI,OK — 2 holder rows for 0xe99cc59f…
7,HISTORICAL order books,unavailable — books were never captured histor...


<a id="findings"></a>
## 7 · Findings, caveats, next steps

* The PM WC universe splits ~1,450 markets into: match 1X2 (282 outcome
  markets → ~91–94 assembled matches, 90-min settlement), advancement
  ladders (240, ET+pens), group/outright futures (~925).
* Trade history is deep and real (2M+ prints) — good enough to price
  convergence and in-play behaviour without any quote reconstruction.
* Live books/gamma depend on the VPN route; the coverage table above
  records exactly what this run could and couldn't see.
* **Next:** notebook 03 matches this universe against the Odds API side;
  04/05 consume `pm_trades` + `pm_match_events` for convergence/in-play.